# Fig 5 supp_1 - library / data QC (curated cohort, n=71)

Publication-quality QC establishing that (i) the libraries are deeply and adequately sequenced, and
(ii) **ASD and control samples are technically comparable** - sequencing depth, gene detection and
APA-feasibility do not differ between groups - so the blood signature (5B) and classifier (5D) are
not driven by a QC/depth artifact. APA is robustly profiled (multi-UTR genes per sample). Two-sided
Mann-Whitney U per metric (ASD vs CTRL). Sources: true gene-level counts
(`all_till_06Jul/.../gene_counts.h5ad`, 17,560 genes) for depth/detection/saturation; the APA
feature table (`...genelevel_bw12_above10cts_apa_named.h5ad`, curated) for the multi-UTR metric.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import anndata as ad, pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

sys.path.insert(0, f"{REPO_ROOT}/figures/style")
import paper_fig_style as S

S.apply_rcparams()
PANEL = "fig_s5a_qc"
ASD, CTRL = "#d73027", "#4575b4"
COL = {"ASD": ASD, "CTRL": CTRL}

# --- true gene-level counts (curated cohort) -> depth, detection, saturation ---
q = ad.read_h5ad(
    f"{DATA_ROOT}/3utr/Blood/all_till_06Jul/results/results_blood_3UTR_till_06jul_gene_counts.h5ad"
)
Xq = q.X
Xq = Xq.toarray() if hasattr(Xq, "toarray") else np.asarray(Xq)
dq = q.obs["Diagnosis"].astype(str).values
lib = Xq.sum(1) / 1e6  # million counts / sample
ndet = (Xq > 0).sum(1) / 1e3  # thousand genes detected / sample

# --- APA feature table -> multi-UTR (APA-testable) genes per sample ---
g = ad.read_h5ad(
    f"{DATA_ROOT}/3utr/Blood/all_till_20240724/utrcalling/UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"
)
cur = g.obs.status_split.astype(str).isin(["Train", "Test_original"]).values
Xg = g.X
Xg = Xg.toarray() if hasattr(Xg, "toarray") else np.asarray(Xg)
Xg = Xg[cur, :]
dgn = g.obs["Diagnosis"].astype(str).values[cur]
gene = g.var["name"].astype(str).values
THR = 5
pres = pd.DataFrame(
    (Xg >= THR).T, index=gene
)  # features x samples (boolean), grouped by gene
nutr = pres.groupby(level=0).sum()  # genes x samples = # expressed UTRs
multi = (nutr >= 2).sum(axis=0).values / 1e3  # thousand multi-UTR genes / sample


def mw(vals, dx):
    a = vals[dx == "ASD"]
    c = vals[dx == "CTRL"]
    p = mannwhitneyu(a, c, alternative="two-sided").pvalue
    return p, ("n.s." if p >= 0.05 else f"p={p:.1e}")


for nm, v, dx in [
    ("library size (M)", lib, dq),
    ("genes det (k)", ndet, dq),
    ("multi-UTR genes (k)", multi, dgn),
]:
    p, lab = mw(v, dx)
    print(
        f"{nm:22s} ASD med {np.median(v[dx=='ASD']):.2f} vs CTRL {np.median(v[dx=='CTRL']):.2f}  MW {lab}"
    )

In [ ]:
def boxstrip(ax, vals, dx, ylab, fmt="%.0f"):
    groups = ["CTRL", "ASD"]
    data = [vals[dx == k] for k in groups]
    bp = ax.boxplot(
        data,
        positions=[0, 1],
        widths=0.55,
        showfliers=False,
        patch_artist=True,
        medianprops=dict(color="black", lw=1.0),
        whiskerprops=dict(lw=0.7),
        capprops=dict(lw=0.7),
        boxprops=dict(lw=0.7),
    )
    for patch, k in zip(bp["boxes"], groups):
        patch.set_facecolor(COL[k])
        patch.set_alpha(0.28)
        patch.set_edgecolor(COL[k])
    rng = np.random.default_rng(0)
    for i, k in enumerate(groups):
        y = vals[dx == k]
        x = rng.normal(i, 0.07, len(y))
        ax.scatter(x, y, s=8, c=COL[k], alpha=0.85, edgecolors="none", zorder=3)
    p, lab = mw(vals, dx)
    ymax = max(np.max(d) for d in data)
    ax.text(0.5, ymax * 1.02, lab, ha="center", fontsize=S.FONT_ANNOT)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        [f"CTRL\n(n={int((dx=='CTRL').sum())})", f"ASD\n(n={int((dx=='ASD').sum())})"],
        fontsize=S.FONT_TICK,
    )
    ax.set_ylabel(ylab, fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    [s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]


fig, ax = plt.subplots(
    1,
    4,
    figsize=(S.FULL_W_PX / 300, S.mm_to_px(52) / 300),
    gridspec_kw={"wspace": 0.46},
)
boxstrip(ax[0], lib, dq, r"library size ($\times10^6$ counts)")
boxstrip(ax[1], ndet, dq, r"genes detected ($\times10^3$)")
boxstrip(ax[2], multi, dgn, r"multi-UTR APA genes ($\times10^3$)")
# (d) saturation: detection vs depth, by diagnosis
a3 = ax[3]
for k in ["CTRL", "ASD"]:
    a3.scatter(
        lib[dq == k],
        ndet[dq == k],
        s=12,
        c=COL[k],
        alpha=0.8,
        edgecolors="none",
        label=k,
    )
a3.set_xlabel(r"library size ($\times10^6$)", fontsize=S.FONT_LABEL)
a3.set_ylabel(r"genes detected ($\times10^3$)", fontsize=S.FONT_LABEL)
a3.set_title("sequencing saturation", fontsize=S.FONT_ANNOT)
a3.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="lower right")
a3.tick_params(labelsize=S.FONT_TICK)
[s.set_visible(False) for s in (a3.spines["top"], a3.spines["right"])]
for lab, a_ in zip("abcd", ax):
    a_.text(
        -0.30,
        1.04,
        lab,
        transform=a_.transAxes,
        fontsize=S.FONT_LABEL + 1,
        fontweight="bold",
    )
fig.subplots_adjust(left=0.055, right=0.99, bottom=0.17, top=0.92)
S.save_and_show(fig, PANEL, figure="Fig5/supp_1")

### Size variants: quarter-height full-width (1×4) and quarter-height half-width (2×2)


In [ ]:
# --- Size variants of fig_s5a_qc: quarter-height full-width (1x4) + quarter-height half-width (2x2) ---
def _qc_full(figsize, save_name):
    fig, ax = plt.subplots(1, 4, figsize=figsize, gridspec_kw={"wspace": 0.46})
    boxstrip(ax[0], lib, dq, r"library size ($\times10^6$ counts)")
    boxstrip(ax[1], ndet, dq, r"genes detected ($\times10^3$)")
    boxstrip(ax[2], multi, dgn, r"multi-UTR APA genes ($\times10^3$)")
    a3 = ax[3]
    for k in ["CTRL", "ASD"]:
        a3.scatter(
            lib[dq == k],
            ndet[dq == k],
            s=12,
            c=COL[k],
            alpha=0.8,
            edgecolors="none",
            label=k,
        )
    a3.set_xlabel(r"library size ($\times10^6$)", fontsize=S.FONT_LABEL)
    a3.set_ylabel(r"genes detected ($\times10^3$)", fontsize=S.FONT_LABEL)
    a3.set_title("sequencing saturation", fontsize=S.FONT_ANNOT)
    a3.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="lower right")
    a3.tick_params(labelsize=S.FONT_TICK)
    [s.set_visible(False) for s in (a3.spines["top"], a3.spines["right"])]
    for lab, a_ in zip("abcd", ax):
        a_.text(
            -0.30,
            1.04,
            lab,
            transform=a_.transAxes,
            fontsize=S.FONT_LABEL + 1,
            fontweight="bold",
        )
    fig.subplots_adjust(left=0.055, right=0.99, bottom=0.17, top=0.92)
    S.save_and_show(fig, save_name, figure="Fig5/supp_1")


def _qc_compact(figsize, save_name):
    def bs(ax, vals, dx, ylab):
        groups = ["CTRL", "ASD"]
        data = [vals[dx == k] for k in groups]
        bp = ax.boxplot(
            data,
            positions=[0, 1],
            widths=0.55,
            showfliers=False,
            patch_artist=True,
            medianprops=dict(color="black", lw=1.0),
            whiskerprops=dict(lw=0.7),
            capprops=dict(lw=0.7),
            boxprops=dict(lw=0.7),
        )
        for patch, k in zip(bp["boxes"], groups):
            patch.set_facecolor(COL[k])
            patch.set_alpha(0.28)
            patch.set_edgecolor(COL[k])
        r = np.random.default_rng(0)
        for i, k in enumerate(groups):
            yy = vals[dx == k]
            xx = r.normal(i, 0.07, len(yy))
            ax.scatter(xx, yy, s=6, c=COL[k], alpha=0.85, edgecolors="none", zorder=3)
        p, lab = mw(vals, dx)
        ymax = max(np.max(d) for d in data)
        ax.text(0.5, ymax * 1.03, lab, ha="center", fontsize=S.FONT_ANNOT - 0.5)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["CTRL", "ASD"], fontsize=S.FONT_TICK)
        ax.set_ylabel(ylab, fontsize=S.FONT_LABEL - 0.5)
        ax.tick_params(labelsize=S.FONT_TICK)
        [s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]

    fig, ax = plt.subplots(2, 2, figsize=figsize)
    ax = ax.ravel()
    bs(ax[0], lib, dq, r"library (×10$^6$)")
    bs(ax[1], ndet, dq, r"genes det. (×10$^3$)")
    bs(ax[2], multi, dgn, r"multi-UTR (×10$^3$)")
    a3 = ax[3]
    for k in ["CTRL", "ASD"]:
        a3.scatter(
            lib[dq == k],
            ndet[dq == k],
            s=8,
            c=COL[k],
            alpha=0.8,
            edgecolors="none",
            label=k,
        )
    a3.set_xlabel(r"library (×10$^6$)", fontsize=S.FONT_LABEL - 0.5)
    a3.set_ylabel(r"genes det. (×10$^3$)", fontsize=S.FONT_LABEL - 0.5)
    a3.set_title("saturation", fontsize=S.FONT_ANNOT)
    a3.legend(
        fontsize=S.FONT_LEGEND - 0.5,
        frameon=False,
        loc="lower right",
        handletextpad=0.3,
        labelspacing=0.2,
    )
    a3.tick_params(labelsize=S.FONT_TICK)
    [s.set_visible(False) for s in (a3.spines["top"], a3.spines["right"])]
    for lab, (fx, fy) in zip(
        "abcd", [(0.005, 0.985), (0.515, 0.985), (0.005, 0.475), (0.515, 0.475)]
    ):
        fig.text(
            fx, fy, lab, fontsize=S.FONT_LABEL, fontweight="bold", va="top", ha="left"
        )
    fig.subplots_adjust(
        left=0.12, right=0.98, bottom=0.16, top=0.92, wspace=0.5, hspace=0.62
    )
    S.save_and_show(fig, save_name, figure="Fig5/supp_1")


_qc_full((S.FULL_W_PX / 300, S.QUARTER_H_PX / 300), "fig_s5a_qc__quarter_full")
_qc_compact((S.HALF_W_PX / 300, S.QUARTER_H_PX / 300), "fig_s5a_qc__quarter_half")

**Caption (S5a).** Library and APA-profiling QC on the curated cohort (40 ASD / 31 control).
(**a**) Sequencing depth, (**b**) genes detected, and (**c**) multi-UTR (APA-testable) genes profiled
per sample do not differ between ASD and controls (two-sided Mann-Whitney, n.s.), so the blood APA
signature and classifier are not explained by a depth or detection bias between groups. (**d**)
Genes detected versus library size shows adequately and comparably sequenced libraries with no
diagnosis-driven separation in QC space. A multi-UTR gene is one with >=2 UTR isoforms each at >=5
counts in that sample.
